<a href="https://colab.research.google.com/github/odamasceno/Calculo-Numerico/blob/main/atividade_3_28_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Atividade 3 - Cáluclo Numérico 28/09**

Felipe Pereira Vergal Rostirola 185210

Iago da Silva Leite 185228

Rafael Damasceno Marques dos Santos 187561



#**Partes 1 e 2.**

Implementação dos métodos, execução, organização da tabela e explicação dos parâmetros.






In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

**I. Método da Bisseção**

In [ ]:
def bissecao(funcao, a, b, tolerancia=1e-6, max_iter=100):
    # Método da bisseção com registro das iterações.
    if a >= b:
        raise ValueError("É necessário fornecer a < b.")
    if tolerancia <= 0 or max_iter < 1:
        raise ValueError("Use tolerância positiva e max_iter >= 1.")

    fa, fb = funcao(a), funcao(b)
    if not np.isfinite([fa, fb]).all():
        raise ValueError("A função não é finita nas extremidades.")
    if fa == 0:
        return a, pd.DataFrame(), "a é raiz"
    if fb == 0:
        return b, pd.DataFrame(), "b é raiz"
    if fa*fb > 0:
        raise ValueError("Não há mudança de sinal nas extremidades.")

    historico = []
    for k in range(1, max_iter+1):
        m = a + (b-a)/2
        fm = funcao(m)
        erro = (b-a)/2
        historico.append({
            "iteração": k, "a": a, "b": b, "m": m,
            "f(a)": fa, "f(m)": fm, "f(b)": fb,
            "erro máximo": erro
        })

        if fm == 0 or erro <= tolerancia:
            motivo = "raiz exata" if fm == 0 else "tolerância"
            break

        if fa*fm < 0:
            b, fb = m, fm
        else:
            a, fa = m, fm
    else:
        raise RuntimeError("Número máximo de iterações atingido.")

    return m, pd.DataFrame(historico), motivo


**II. Método de Newton**

In [ ]:
def newton(f, df, x0, tol=1e-6, max_iter=100, deriv_tol=1e-14):
    """Resolve f(x)=0 pelo método de Newton e retorna raiz e histórico."""
    x = float(x0)
    historico = []

    for k in range(max_iter):
        fx = f(x)
        dfx = df(x)
        historico.append({"k": k, "x": x, "f(x)": fx, "f'(x)": dfx})

        if abs(fx) <= tol:
            return x, historico, "convergiu pelo resíduo"

        if abs(dfx) <= deriv_tol:
            return x, historico, "parou: derivada nula ou muito pequena"

        x_novo = x - fx / dfx

        # Um passo pequeno sem resíduo pequeno pode indicar estagnação.
        if abs(x_novo - x) <= tol * max(1.0, abs(x_novo)):
            x = x_novo
            if abs(f(x)) <= tol:
                historico.append({"k": k + 1, "x": x, "f(x)": f(x), "f'(x)": df(x)})
                return x, historico, "convergiu pelo resíduo após passo pequeno"
            return x, historico, "parou: passo pequeno, mas resíduo ainda alto"

        x = x_novo

    return x, historico, "parou: atingiu o máximo de iterações"

**III. Método das Secante**

In [ ]:
def secantes(f, x0, x1, tol=1e-6, max_iter=100, denom_tol=1e-14):
    """Resolve f(x)=0 pelo método das secantes."""
    x_anterior = float(x0)
    x_atual = float(x1)
    historico = []

    for k in range(max_iter):
        f_anterior = f(x_anterior)
        f_atual = f(x_atual)
        historico.append({"k": k, "x": x_atual, "f(x)": f_atual})

        if abs(f_atual) <= tol:
            return x_atual, historico, "convergiu pelo resíduo"

        denominador = f_atual - f_anterior
        if abs(denominador) <= denom_tol:
            return x_atual, historico, "parou: denominador nulo ou muito pequeno"

        x_novo = x_atual - f_atual * (x_atual - x_anterior) / denominador

        if abs(x_novo - x_atual) <= tol * max(1.0, abs(x_novo)):
            if abs(f(x_novo)) <= tol:
                historico.append({"k": k + 1, "x": x_novo, "f(x)": f(x_novo)})
                return x_novo, historico, "convergiu pelo resíduo após passo pequeno"
            return x_novo, historico, "parou: passo pequeno, mas resíduo ainda alto"

        x_anterior, x_atual = x_atual, x_novo

    return x_atual, historico, "parou: atingiu o máximo de iterações"


**Funções e Derivadas:**

In [ ]:
def f1(x): return x**2 - 2
def df1(x): return 2*x

def f2(x): return x**3 - x - 2
def df2(x): return 3*(x**2) - 1

def f3(x): return math.exp(-x) - x
def df3(x): return -math.exp(-x) - 1

def f4(x): return x**3 - 2*x + 2
def df4(x): return 3*(x**2) - 2

**Execuções**

In [ ]:
# Configurações globais
TOLERANCIA = 1e-6
MAX_ITERACOES = 100

# Valores dos testes
testes = [
    {"nome": "f1(x) = x^2 - 2", "f": f1, "df": df1, "bis_ab": (1, 2), "newton_x0": 1.5, "sec_x0x1": (1, 2)},
    {"nome": "f2(x) = x^3 - x - 2", "f": f2, "df": df2, "bis_ab": (1, 2), "newton_x0": 1.5, "sec_x0x1": (1, 2)},
    {"nome": "f3(x) = e^(-x) - x", "f": f3, "df": df3, "bis_ab": (0, 1), "newton_x0": 0.5, "sec_x0x1": (0, 1)},
    {"nome": "f4(x) = x^3 - 2x + 2", "f": f4, "df": df4, "bis_ab": (-2, -1), "newton_x0": -2.0, "sec_x0x1": (-2, -1)}
]

resultados = []

for teste in testes:
    nome_f = teste["nome"]
    f = teste["f"]
    df = teste["df"]

    # Teste 1: Bisseção
    a, b = teste["bis_ab"]
    try:
        raiz_b, hist_b, estado_b = bissecao(f, a, b, tolerancia=TOLERANCIA, max_iter=MAX_ITERACOES)
        val_final_b = abs(f(raiz_b))
        iter_b = len(hist_b) # len() num DataFrame retorna o número de linhas (iterações)
    except Exception as e:
        raiz_b = None
        val_final_b = None
        iter_b = "-"
        estado_b = f"Falha: {e}"

    resultados.append({
        "Função": nome_f,
        "Método": "Bisseção",
        "Valores iniciais": f"[{a}, {b}]",
        "Raiz aproximada": raiz_b,
        "|f(x)| final": val_final_b,
        "Iterações/resultado": f"{iter_b} / {estado_b}"
    })


    # Teste 2: Newton
    x0 = teste["newton_x0"]
    raiz_n, hist_n, estado_n = newton(f, df, x0, tol=TOLERANCIA, max_iter=MAX_ITERACOES)
    val_final_n = abs(f(raiz_n)) if raiz_n is not None else None
    iter_n = len(hist_n)

    resultados.append({
        "Função": nome_f,
        "Método": "Newton",
        "Valores iniciais": f"{x0}",
        "Raiz aproximada": raiz_n,
        "|f(x)| final": val_final_n,
        "Iterações/resultado": f"{iter_n} / {estado_n}"
    })


    # Teste 3: Secante
    x0_s, x1_s = teste["sec_x0x1"]
    raiz_s, hist_s, estado_s = secantes(f, x0_s, x1_s, tol=TOLERANCIA, max_iter=MAX_ITERACOES)
    val_final_s = abs(f(raiz_s)) if raiz_s is not None else None
    iter_s = len(hist_s)

    resultados.append({
        "Função": nome_f,
        "Método": "Secante",
        "Valores iniciais": f"({x0_s}, {x1_s})",
        "Raiz aproximada": raiz_s,
        "|f(x)| final": val_final_s,
        "Iterações/resultado": f"{iter_s} / {estado_s}"
    })

# Criação e exibição da tabela final
tabela_analise = pd.DataFrame(resultados)
display(tabela_analise)

,Função,Método,Valores iniciais,Raiz aproximada,|f(x)| final,Iterações/resultado
0,f1(x) = x^2 - 2,Bisseção,"[1, 2]",1.414214,1.617417e-06,20 / tolerância
1,f1(x) = x^2 - 2,Newton,1.5,1.414214,4.510614e-12,4 / convergiu pelo resíduo
2,f1(x) = x^2 - 2,Secante,"(1, 2)",1.414214,8.931456e-10,6 / convergiu pelo resíduo
3,f2(x) = x^3 - x - 2,Bisseção,"[1, 2]",1.521380,4.265829e-06,20 / tolerância
4,f2(x) = x^3 - x - 2,Newton,1.5,1.521380,5.893874e-07,3 / convergiu pelo resíduo
5,f2(x) = x^3 - x - 2,Secante,"(1, 2)",1.521380,7.015478e-09,7 / convergiu pelo resíduo
6,f3(x) = e^(-x) - x,Bisseção,"[0, 1]",0.567143,2.348157e-07,20 / tolerância
7,f3(x) = e^(-x) - x,Newton,0.5,0.567143,1.964805e-07,3 / convergiu pelo resíduo
8,f3(x) = e^(-x) - x,Secante,"(0, 1)",0.567143,2.538017e-08,5 / convergiu pelo resíduo
9,f4(x) = x^3 - 2x + 2,Bisseção,"[-2, -1]",-1.769292,3.521840e-06,20 / tolerância


Para garantir uma convergência confiável, optou-se por utilizar uma combinação de dois critérios em simultâneo. Os algoritmos implementados apenas consideram que o método convergiu com sucesso quando a diferença entre duas aproximações sucessivas, é menor que a tolerância definida e, simultaneamente, o valor absoluto da função na nova aproximação, que corresponde ao resíduo, também é inferior a essa mesma tolerância. O método da bisseção para imediatamente se for detectado que a função avaliada nos extremos do intervalo tem o mesmo sinal. O método de Newton é interrompido caso a derivada se aproxime perigosamente de zero, evitando assim divisões impossíveis. De igual forma, o método da secante aborta o cálculo se o denominador tender para zero. Por fim, todos os métodos incluem um limite máximo de 100 iterações, após o qual a execução é interrompida e sinalizada por ausência de convergência.  

#**Parte 3.**

Criação dos GIFs.






In [ ]:
# Função de Animação - Bisseção
def animar_bissecao(hist_df, f, x_range, filename):
    fig, ax = plt.subplots(figsize=(8, 6))
    x_vals = np.linspace(x_range[0], x_range[1], 400)
    y_vals = [f(x) for x in x_vals]

    def update(frame):
        ax.cla()
        ax.plot(x_vals, y_vals, 'k-', label='f(x)')
        ax.axhline(0, color='gray', linestyle='--')

        # Lê os dados do DataFrame na linha atual
        row = hist_df.iloc[frame]
        k = int(row['iteração'])
        a, b, m = row['a'], row['b'], row['m']
        erro = row['erro máximo']

        ax.axvline(a, color='red', linestyle=':', label='a, b (intervalo)')
        ax.axvline(b, color='red', linestyle=':')
        ax.plot(m, 0, 'bo', label=f'Aproximação m={m:.4f}')
        ax.plot([a, b], [0, 0], 'r-', lw=3, alpha=0.5)

        info = f"Iteração: {k}\nAproximação: {m:.5f}\nErro: {erro:.5e}"
        ax.text(0.05, 0.05, info, transform=ax.transAxes, bbox=dict(facecolor='white', alpha=0.8))
        ax.set_title("Método da Bisseção")
        ax.set_xlim(x_range)
        ax.legend(loc="upper right")

    ani = FuncAnimation(fig, update, frames=len(hist_df), repeat=False)
    ani.save(filename, writer=PillowWriter(fps=1.5))
    plt.close(fig)

# Função de Animação - Newton
def animar_newton(hist_list, f, x_range, filename):
    fig, ax = plt.subplots(figsize=(8, 6))
    x_vals = np.linspace(x_range[0], x_range[1], 400)
    y_vals = [f(x) for x in x_vals]

    def update(frame):
        ax.cla()
        ax.plot(x_vals, y_vals, 'k-', label='f(x)')
        ax.axhline(0, color='gray', linestyle='--')

        estado = hist_list[frame]
        k, x, fx, dfx = estado['k'], estado['x'], estado['f(x)'], estado["f'(x)"]

        ax.plot(x, fx, 'ro', label=f'x_{k}={x:.4f}')
        ax.plot([x, x], [0, fx], 'r:', alpha=0.5)

        if dfx != 0:
            x_prox = x - fx/dfx
            x_tang = np.array([x_prox, x + (x - x_prox)])
            y_tang = dfx * (x_tang - x) + fx
            ax.plot(x_tang, y_tang, 'b--', label='Reta Tangente')
            ax.plot(x_prox, 0, 'go', label=f'x_{k+1}={x_prox:.4f}')

        info = f"Iteração: {k}\nAproximação: {x:.5f}\nResíduo |f(x)|: {abs(fx):.5e}"
        ax.text(0.05, 0.05, info, transform=ax.transAxes, bbox=dict(facecolor='white', alpha=0.8))
        ax.set_title("Método de Newton")
        ax.set_xlim(x_range)
        ax.legend(loc="upper right")

    ani = FuncAnimation(fig, update, frames=len(hist_list), repeat=False)
    ani.save(filename, writer=PillowWriter(fps=1.5))
    plt.close(fig)

# 3. Função de Animação - Secante
def animar_secante(hist_list, f, x0_inicial, x_range, filename):
    fig, ax = plt.subplots(figsize=(8, 6))
    x_vals = np.linspace(x_range[0], x_range[1], 400)
    y_vals = [f(x) for x in x_vals]

    def update(frame):
        ax.cla()
        ax.plot(x_vals, y_vals, 'k-', label='f(x)')
        ax.axhline(0, color='gray', linestyle='--')

        estado = hist_list[frame]
        k, x, fx = estado['k'], estado['x'], estado['f(x)']

        # Recupera o x anterior (se for a primeira iteração, usa o x0 inicial)
        x_ant = x0_inicial if frame == 0 else hist_list[frame-1]['x']
        f_ant = f(x_ant)

        ax.plot(x_ant, f_ant, 'ro', label=f'x_{k-1}')
        ax.plot(x, fx, 'mo', label=f'x_{k}')
        ax.plot([x_ant, x_ant], [0, f_ant], 'r:', alpha=0.3)
        ax.plot([x, x], [0, fx], 'm:', alpha=0.3)

        if fx - f_ant != 0:
            x_prox = x - fx * (x - x_ant) / (fx - f_ant)
            x_sec = np.array([x_prox, x_ant])
            y_sec = f_ant + (fx - f_ant)/(x - x_ant) * (x_sec - x_ant)
            ax.plot(x_sec, y_sec, 'b--', label='Reta Secante')
            ax.plot(x_prox, 0, 'go', label=f'x_{k+1}')

        info = f"Iteração: {k}\nAproximação: {x:.5f}\nResíduo |f(x)|: {abs(fx):.5e}"
        ax.text(0.05, 0.05, info, transform=ax.transAxes, bbox=dict(facecolor='white', alpha=0.8))
        ax.set_title("Método das Secantes")
        ax.set_xlim(x_range)
        ax.legend(loc="upper right")

    ani = FuncAnimation(fig, update, frames=len(hist_list), repeat=False)
    ani.save(filename, writer=PillowWriter(fps=1.5))
    plt.close(fig)

# Geração dos GIFs
print("A gerar os ficheiros GIF...")

# Bisseção com f1(x) = x^2 - 2, intervalo [1, 2]
_, hist_b, _ = bissecao(f1, 1, 2)
animar_bissecao(hist_b, f1, (0.5, 2.5), "bissecao_f1.gif")

# Newton com f3(x) = e^(-x) - x, x0 = 0.5
_, hist_n, _ = newton(f3, df3, 0.5)
animar_newton(hist_n, f3, (-0.2, 1.2), "newton_f3.gif")

# Secante com f2(x) = x^3 - x - 2, x0=1, x1=2
_, hist_s, _ = secantes(f2, 1, 2)
animar_secante(hist_s, f2, 1, (0.5, 2.5), "secante_f2.gif")

print("Geração concluída com sucesso! Os ficheiros .gif encontram-se no diretório atual.")

A gerar os ficheiros GIF...
Geração concluída com sucesso! Os ficheiros .gif encontram-se no diretório atual.


https://www.mathcha.io/editor/6rrDMCwVIxzFgJtq7JOLEuM8BvQdh1eJ2gEHlWPYDr